# ENG-209, Partie 2, Série 9

## Classification par Régression Logistique

Dans cet exercice, vous étudierez les données relatives aux ressources humaines afin de découvrir les facteurs qui conduisent à l'attrition des employés. L'objectif est de créer un modèle de régression logistique pour prédire le départ des employés.

#### À propos des données

Les données de cet exercice sont issues du projet Kaggle [_IBM HR Analytics Employee Attrition & Performance_](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset).

Pour vous faciliter la tâche, nous avons déjà préparé les données pour les utiliser avec des modèles de régression logistique.

Les données sont disponible dans le fichier `../data/hr/hr-churn-1.csv`

#### Énoncé d'exercice

Pour cet exercice nous vous demandons:
1. D'explorer et comprendre les données mises à votre disposition.
2. D'utiliser certaines de ces données pour créer un modèle de [régression logistique](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) qui vous permette d'estimer la probabilité de départ des employés.
3. De vérifier la qualité de votre modèle et de vous assurer qu'il est généralisable.
4. Ajuster le threshold pour faire varier les false positives vs false négatives (voir [predict_proba](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba)).

💡 **Astuces**

* Utilisez les méthodes pandas pour lire et comprendre les données.
* Utilisez les bibliothèques scikit-learn, pour entrainer votre model:
    - sklearn.linear_model.LogisticRegression
    - sklearn.model_selection.train_test_split
    - sklearn.preprocessing.StandardScaler
    - sklearn.pipeline.Pipeline
* Assurez-vous que vos données de test sont représentatives de vos données d'apprentissage. En particulier, veillez à ce que le taux d'attrition soit respecté dans les données d'entrainment et de test (voir les paramètres de la documentation `train_test_split`).
* N'hesitez pas à jouer avec les paramètres de la LogisticRegression pour améliorer vos résultats (mais veuillez bien lire la [documentation](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression))
* Utilisez les bibliothèques scikit-learn pour vérifier votre modèle:
    - sklearn.metrics.confusion_matrix
    - sklearn.metrics.classification_report
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from numpy.typing import NDArray
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
hr_churn_df: pd.DataFrame = pd.read_csv('../data/hr/hr-churn-1.csv')

In [ ]:
# A vous de jouer!

### Exploration des données

In [ ]:
hr_churn_df.info()

In [ ]:
hr_churn_df.describe()

In [ ]:
hr_churn_df.corr()

Pour vous aider, nous vous proposons une ébauche de fonction (à compléter) permettant de comparer des histogrammes de différentes variables entre les employés qui restent et ceux qui partent. N’hésitez pas à l’améliorer, par exemple en ajoutant une courbe représentant le pourcentage de départs pour chaque « bin » de l’histogramme.

In [ ]:
def compare_histogram(df: pd.DataFrame, variable: str):
    fig, ax = plt.subplots()

    ax.hist(x=df[df.Attrition < 1][variable], color='red', alpha=0.5, label='Restent')
    ax.hist(x=df[df.Attrition > 0][variable], color='blue', alpha=0.5, label="Partent")

    ax.set_title(f"Taux de départs en fonction de {variable}")
    ax.set_xlabel(variable)
    ax.set_ylabel("Fréquence")

    ax.legend(loc='best')

    plt.show()

In [ ]:
compare_histogram(hr_churn_df, "Age")

In [ ]:
compare_histogram(hr_churn_df, "JobSatisfaction")

In [ ]:
compare_histogram(hr_churn_df, "TotalWorkingYears")

In [ ]:
compare_histogram(hr_churn_df, "YearsInCurrentRole")

In [ ]:
compare_histogram(hr_churn_df, "JobLevel")

In [ ]:
compare_histogram(hr_churn_df, "MonthlyIncome")

#### Commentaires

- Le jeu de données comporte 1470 observations (employés).

- Le jeu de données ne contient aucune valeur manquante.

- Toutes les variables sont numériques.

- Les valeurs (âge, charge de travail, salaires, ...) se situent dans des intervalles raisonnables.

- Standard hours est de $80$ pour tous les employés, cette colonne n'apporte donc pas d'information.

- L’âge (et donc l’ancienneté), la satisfaction au travail, le nombre d’années passées dans l’entreprise et dans le poste actuel, le niveau hiérarchique ainsi que le salaire apparaissent comme des facteurs explicatifs importants du taux de départ. Notez cependant que certaines de ces variables peuvent être fortement corrélées entre elles, par exemple l’âge et le nombre total d’années travaillées, ou encore le poste et le salaire.

- Le taux d’attrition (0: reste, 1: part) est déséquilibré (≈16 %) : il faudra donc veiller, lors de la séparation train/test, **à utiliser un échantillonnage stratifié** afin de conserver une répartition représentative dans les deux ensembles.

### Modélisation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report

#### Split Train/Test

Utilisons 60% des données pour l'apprentissage du model de régression logistique et le reste pour tester le model (test_size=0.4)

In [ ]:
X: NDArray[np.float64] = hr_churn_df.loc[:,~hr_churn_df.columns.isin(['Attrition'])].to_numpy()
y: NDArray[np.float64] = hr_churn_df['Attrition'].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)

#### Apprentissage

Essayons de varier les paramètres:
* solver: `newton-cg`, `lbfgs`, `liblinear`, `sag`, `saga`
* penalty: `l1`, `l2`, `elasticnet`, `None`, ...
* regularisation: `C`

In [ ]:
model = Pipeline(
    [
        ('scaler', StandardScaler()),
        ('logistic', LogisticRegression(solver='liblinear', l1_ratio=1, C=1.0))
    ])

In [ ]:
model.fit(X_train, y_train)

Variables indépendentes utilisées

### Validation du modèle de régression

In [ ]:
y_pred: NDArray[np.float64] = np.asarray(model.predict(X_test))

In [ ]:
y_pred[y_test==y_pred].shape[0]/y_pred.shape[0]

In [ ]:
model.score(X_test,y_test)

In [ ]:
confusion_matrix(y_test,y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
plt.rcParams["figure.figsize"] = (6, 4)

cm: NDArray[np.float64] = confusion_matrix(y_test, y_pred)
disp: ConfusionMatrixDisplay = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['restent','partent'])

disp.plot()
plt.show()

In [ ]:
print(f'{classification_report(y_true=y_test,y_pred=y_pred, target_names=['restent', 'partent'])}')

---
En régression logistique, l'exponentiation du coefficient $e^{\beta_j}$ (odds ratio) est un ratio indiquant de combien la prédiction change lorsqu'une charactéristique augmente d'une unité. Une valeur égale ou proche de 1 indique que la caractéristique a peu ou pas d'effet sur le modèle. Notez qu'il peut être pertinent de réentraîner le modèle après avoir retiré ces caractéristiques. Nous explorerons par la suite des algorithmes de scikit-learn, tels que RFE ou RFECV, qui automatisent ce processus.

Nous associons ces valeurs aux colonnes de la matrice des caractéristiques X afin d'en faciliter l'interprétation.

In [ ]:
coefs: NDArray[np.float64] = np.dstack((np.exp(model.named_steps['logistic'].coef_)[0], hr_churn_df.columns[:-1].to_numpy()))[0]
coefs.sort(axis=0)
coefs

Faisons maintenant varier le threshold de detection

In [ ]:
threshold: float = 0.3
probs: NDArray[np.float64] = model.predict_proba(X_test)
y_pred_t: NDArray[np.int8] = (probs[:, 1] >= threshold).astype(np.int8)

In [ ]:
print(f'{classification_report(y_true=y_test,y_pred=y_pred_t, target_names=['restent', 'partent'])}')

In [ ]:
cm = confusion_matrix(y_true=y_test, y_pred=y_pred_t)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['restent', 'partent'])

disp.plot()
plt.show()